# Spark connect to server

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://localhost").getOrCreate()
spark

# Load a CSV file

In [3]:
import os
os.getcwd()

'/home/analytics4220/Yangchen'

In [8]:
df = spark.read.csv("/home/analytics4220/Yangchen/obesity.csv", 
                    header=True, quote='"')
df

DataFrame[YearStart: string, YearEnd: string, LocationAbbr: string, LocationDesc: string, Datasource: string, Class: string, Topic: string, Question: string, Data_Value_Unit: string, Data_Value_Type: string, Data_Value: string, Data_Value_Alt: string, Data_Value_Footnote_Symbol: string, Data_Value_Footnote: string, Low_Confidence_Limit: string, High_Confidence_Limit : string, Sample_Size: string, Total: string, Age(years): string, Education: string, Sex: string, Income: string, Race/Ethnicity: string, GeoLocation: string, ClassID: string, TopicID: string, QuestionID: string, DataValueTypeID: string, LocationID: string, StratificationCategory1: string, Stratification1: string, StratificationCategoryId1: string, StratificationID1: string]

# DataSet

In [42]:
df.show(5)
df.printSchema()

+---------+-------+------------+------------+--------------------+--------------------+--------------------+--------------------+---------------+---------------+----------+--------------+--------------------------+-------------------+--------------------+----------------------+-----------+-----+----------+---------+----+------------------+--------------+--------------------+-------+-------+----------+---------------+----------+-----------------------+------------------+-------------------------+-----------------+
|YearStart|YearEnd|LocationAbbr|LocationDesc|          Datasource|               Class|               Topic|            Question|Data_Value_Unit|Data_Value_Type|Data_Value|Data_Value_Alt|Data_Value_Footnote_Symbol|Data_Value_Footnote|Low_Confidence_Limit|High_Confidence_Limit |Sample_Size|Total|Age(years)|Education| Sex|            Income|Race/Ethnicity|         GeoLocation|ClassID|TopicID|QuestionID|DataValueTypeID|LocationID|StratificationCategory1|   Stratification1|Stratif

# Select one column 

In [43]:
df.select("LocationDesc").show()

+------------+
|LocationDesc|
+------------+
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
|     Alabama|
+------------+
only showing top 20 rows


# Select column + calculation

Add 1 to obesity rate

In [44]:
df.select(
    "LocationDesc",
    (col("Data_Value") + 1).alias("Obesity_Plus_1")
).show()


+------------+--------------+
|LocationDesc|Obesity_Plus_1|
+------------+--------------+
|     Alabama|          35.8|
|     Alabama|          36.8|
|     Alabama|          33.3|
|     Alabama|          35.1|
|     Alabama|          29.8|
|     Alabama|          17.3|
|     Alabama|          28.8|
|     Alabama|          36.2|
|     Alabama|          36.5|
|     Alabama|          39.0|
|     Alabama|          37.4|
|     Alabama|          28.1|
|     Alabama|          33.9|
|     Alabama|          NULL|
|     Alabama|          27.4|
|     Alabama|          24.8|
|     Alabama|          32.8|
|     Alabama|          NULL|
|     Alabama|          33.8|
|     Alabama|          29.6|
+------------+--------------+
only showing top 20 rows


# Filter Rows

States with obesity above 35%

In [45]:
df.filter(
    col("Data_Value") > 35
).select(
    "LocationDesc",
    "Data_Value"
).show()

+------------+----------+
|LocationDesc|Data_Value|
+------------+----------+
|     Alabama|      35.8|
|     Alabama|      35.2|
|     Alabama|      35.5|
|     Alabama|      38.0|
|     Alabama|      36.4|
|     Alabama|      38.5|
|     Alabama|      40.1|
|      Alaska|      35.5|
|     Arizona|      36.2|
|    Arkansas|      37.2|
|    Arkansas|      36.4|
|    Arkansas|      36.9|
|    Arkansas|      38.1|
|  California|      37.3|
| Connecticut|      36.0|
|    Delaware|      35.6|
|    Delaware|      36.5|
|    Delaware|      36.1|
|    Delaware|      35.2|
|    Delaware|      38.2|
+------------+----------+
only showing top 20 rows


# GroupBy + Aggregate 

Average obesity by state

In [46]:
df.groupBy("LocationDesc").agg(
    avg("Data_Value").alias("Avg_Obesity")
).show()

+--------------------+------------------+
|        LocationDesc|       Avg_Obesity|
+--------------------+------------------+
|                Utah|30.627204703367184|
|              Hawaii| 31.65583511777302|
|           Minnesota|31.524567836563644|
|                Ohio| 32.30430672268908|
|            National|31.833233979135617|
|            Arkansas| 32.73612188365651|
|              Oregon|31.708008776741632|
|               Texas| 31.90073684210526|
|        North Dakota|31.893548387096775|
|        Pennsylvania|  30.7964438994482|
|         Connecticut|31.283279914529917|
|            Nebraska| 31.86951672862454|
|             Vermont|31.137291066282415|
|              Nevada| 32.08590859630033|
|         Puerto Rico|32.995906432748534|
|          Washington|31.363898734177212|
|            Illinois| 31.77597330367074|
|            Oklahoma|31.850999459751492|
|      Virgin Islands| 33.23910386965375|
|District of Columbia|29.596736725663714|
+--------------------+------------

# SQL Query 

In [47]:
df.createOrReplaceTempView("obesity")

sqlDF = spark.sql("""
SELECT LocationDesc, Data_Value
FROM obesity
WHERE Data_Value > 35
""")

sqlDF.show()

+------------+----------+
|LocationDesc|Data_Value|
+------------+----------+
|     Alabama|      35.8|
|     Alabama|      35.2|
|     Alabama|      35.5|
|     Alabama|      38.0|
|     Alabama|      36.4|
|     Alabama|      38.5|
|     Alabama|      40.1|
|      Alaska|      35.5|
|     Arizona|      36.2|
|    Arkansas|      37.2|
|    Arkansas|      36.4|
|    Arkansas|      36.9|
|    Arkansas|      38.1|
|  California|      37.3|
| Connecticut|      36.0|
|    Delaware|      35.6|
|    Delaware|      36.5|
|    Delaware|      36.1|
|    Delaware|      35.2|
|    Delaware|      38.2|
+------------+----------+
only showing top 20 rows
